# NeqSim CO2 Impurity Kinetics & Thermodynamics Interactive Guide

> **Experimental screening model.** The reaction parameters and fallback thermodynamic
> correlation in this notebook are illustrative and uncalibrated. Do not use the numerical
> predictions as qualified engineering or design data.

This notebook demonstrates chemical-reaction screening for trace impurities in dense-phase and
supercritical CO2 streams.

### Topics covered

1. Model initialization
2. NeqSim SRK thermodynamics with an explicit screening fallback
3. Impurity levels, water content, and wall material
4. Dynamic ODE simulations and summary tables
5. Illustrative sensitivity cases


## 1. Importing NeqSim Impurity Kinetics Framework


In [1]:
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from neqsim_co2_kinetics import CO2ImpurityKineticsModel

warnings.filterwarnings(
    "ignore",
    message="NeqSim Python thermodynamics is unavailable*",
    category=RuntimeWarning,
)
print("CO2 impurity kinetics tutorial module imported.")


CO2 impurity kinetics tutorial module imported.


## 2. Thermodynamic backend and fluid density

The framework uses the **NeqSim SRK equation of state** when the NeqSim Python package is
available. If it is unavailable, the module emits a warning and uses an illustrative screening
correlation so the tutorial remains executable. The fallback is not a replacement for a NeqSim
flash or experimental validation.

The following comparison evaluates pipeline conditions ($25^\circ	ext{C}$, $100	ext{ bar}$)
and ship-transport conditions ($-25^\circ	ext{C}$, $25	ext{ bar}$).


In [2]:
# Pipeline transport (25 °C, 100 bar)
model_pipe = CO2ImpurityKineticsModel(
    T_kelvin=298.15,
    P_bar=100.0,
    water_ppm=50.0,
)
print(f"Pipeline molar density (25 °C, 100 bar): {model_pipe.molar_density:.2f} kmol/m³")
print(f"Pipeline thermodynamic backend: {model_pipe.thermodynamic_backend}")

# Ship transport (-25 °C, 25 bar)
model_ship = CO2ImpurityKineticsModel(
    T_kelvin=248.15,
    P_bar=25.0,
    water_ppm=50.0,
)
print(f"Ship molar density (-25 °C, 25 bar): {model_ship.molar_density:.2f} kmol/m³")
print(f"Ship thermodynamic backend: {model_ship.thermodynamic_backend}")


Pipeline molar density (25 °C, 100 bar): 20.66 kmol/m³
Pipeline thermodynamic backend: illustrative screening correlation
Ship molar density (-25 °C, 25 bar): 24.03 kmol/m³
Ship thermodynamic backend: illustrative screening correlation


## 3. Setting Impurity Levels & Choosing Wall Material

You can specify arbitrary trace impurity concentrations in **parts per million (ppm)**:
- `H2S`: Hydrogen Sulfide (ppm)
- `SO2`: Sulfur Dioxide (ppm)
- `NO2`: Nitrogen Dioxide (ppm)
- `O2`: Molecular Oxygen (ppm)
- `H2O`: Water Content (ppm)

### Wall Material Options:
- `'carbon_steel'` or `'magnetite'`: Catalyzes heterogeneous elemental sulfur formation ($R_8$: $E_{a, \text{S8}} = 42.0\text{ kJ/mol}$).
- `'stainless_steel'` or `'inert'`: Uncatalyzed surface ($E_{a, \text{S8}} = 65.0\text{ kJ/mol}$).



In [3]:
# Define custom feed stream in ppm
custom_feed = {
    'H2S': 10.0,
    'SO2': 10.0,
    'NO2': 10.0,
    'O2': 10.0,
    'H2O': 10.0
}

# Select Carbon Steel / Magnetite Surface
model = CO2ImpurityKineticsModel(
    T_kelvin=248.15,   # -25 °C
    P_bar=25.0,        # 25 bar
    water_ppm=10.0,    # 10 ppm H2O
    material='carbon_steel'
)

print(f"Model initialized for material: '{model.material}' at T={model.T} K, P={model.P} bar")



Model initialized for material: 'carbon_steel' at T=248.15 K, P=25.0 bar


## 4. Helper Functions for Generating Table 1 & Table 2

Below are standard helper functions to simulate any case and format both **Table 1 (Time-Series)** and **Table 2 (Reaction Kinetics & $K_{\text{eq}}$)**.



In [4]:
def generate_tables_for_case(
    case_name,
    feed,
    T_K=248.15,
    P_bar=25.0,
    material="carbon_steel",
):
    model = CO2ImpurityKineticsModel(
        T_kelvin=T_K,
        P_bar=P_bar,
        water_ppm=feed.get("H2O", 10.0),
        material=material,
    )
    rho_m = model.molar_density

    # Dynamic ODE integration over 10 hours.
    result = model.simulate(feed, duration_sec=10.0 * 3600.0, num_points=201)
    time_hours = result["time_hours"]
    target_hours = np.arange(0.0, 11.0, 1.0)

    table1_rows = []
    for target in target_hours:
        row = {"Time (h)": target}
        for species in model.SPECIES:
            value = np.interp(target, time_hours, result["ppm"][species])
            row[f"{species} (ppm)"] = max(0.0, float(value))
        table1_rows.append(row)
    table1 = pd.DataFrame(table1_rows).round(6)

    rates = model.get_reaction_rates(feed.get("H2O", 10.0))
    r8_key = "R8_cs" if material in {"carbon_steel", "magnetite"} else "R8_ss"
    reaction_info = [
        ("R1", "SO2 + 0.5 O2 + H2O -> H2SO4", "R1", "k1_f", "Keq1"),
        ("R2", "H2S + 3 NO2 -> SO2 + H2O + 3 NO", "R2", "k2_f", "Keq2"),
        ("R3A", "SO2 + NO2 + H2O -> H2SO4 + NO", "R3a", "k3a_f", "Keq3"),
        (
            "R3B",
            "SO2 + 0.5 O2 + H2O -> H2SO4 (H2S/NO2 co-catalysed)",
            "R3b",
            "k3b_f",
            None,
        ),
        ("R4", "2 NO + O2 -> 2 NO2", "R4", "k4_f", "Keq4"),
        ("R5", "3 NO2 + H2O -> 2 HNO3 + NO", "R5", "k5_f", "Keq5"),
        ("R6", "H2S + 1.5 O2 -> SO2 + H2O", "R6", "k6_f", "Keq6"),
        ("R7", "5 H2S + 6 NO + 4 H2O -> 6 NH3 + 5 SO2", "R7", "k7_f", None),
        ("R8", "H2S + 0.5 O2 -> 1/8 S8 + H2O", r8_key, "k8_f", None),
    ]

    def fugacity_concentration(species):
        ppm = max(float(feed.get(species, 0.0)), 0.0)
        return ppm * 1.0e-6 * rho_m * model.phi_dict[species]

    c_h2s = fugacity_concentration("H2S")
    c_so2 = fugacity_concentration("SO2")
    c_no2 = fugacity_concentration("NO2")
    c_no = fugacity_concentration("NO")
    c_o2 = fugacity_concentration("O2")
    c_h2o = fugacity_concentration("H2O")

    initial_rates = {
        "R1": rates["k1_f"] * c_so2 * c_o2**0.5 * c_h2o,
        "R2": rates["k2_f"] * c_h2s * c_no2,
        "R3A": rates["k3a_f"] * c_so2 * c_no2 * c_h2o,
        "R3B": rates["k3b_f"] * c_so2 * c_o2**0.5 * c_h2s**0.5 * c_no2,
        "R4": rates["k4_f"] * c_no**2 * c_o2,
        "R5": rates["k5_f"] * c_no2**3 * c_h2o,
        "R6": rates["k6_f"] * c_h2s * c_o2**1.5,
        "R7": rates["k7_f"] * c_h2s * c_no * c_h2o,
        "R8": rates["k8_f"] * c_h2s * c_o2**0.5,
    }

    table2_rows = []
    for reaction_id, equation, parameter_key, rate_key, equilibrium_key in reaction_info:
        rate = initial_rates[reaction_id]
        equilibrium = (
            "irreversible"
            if equilibrium_key is None
            else f"{rates[equilibrium_key]:.4e}"
        )
        table2_rows.append(
            {
                "ID": reaction_id,
                "Net reaction": equation,
                "Ea (kJ/mol)": model.kinetic_params[parameter_key]["Ea"] / 1000.0,
                "Keq": equilibrium,
                "k_forward": f"{rates[rate_key]:.4e}",
                "initial rate (kmol/m3/s)": f"{rate:.4e}",
                "initial rate (ppm/h)": f"{(rate / rho_m) * 1.0e6 * 3600.0:.4e}",
            }
        )
    table2 = pd.DataFrame(table2_rows)

    print("=" * 100)
    print(
        f"{case_name.upper()} "
        f"(T={T_K - 273.15:.1f} °C, P={P_bar} bar, material={material})"
    )
    print("=" * 100)
    print("\nTABLE 1: ILLUSTRATIVE 10-HOUR SPECIES CONCENTRATIONS")
    display(table1)
    print("\nTABLE 2: ILLUSTRATIVE RATE-LAW SUMMARY")
    display(table2)

    return table1, table2


## 5. Illustrative sensitivity cases

These cases exercise the API and show qualitative parameter sensitivity. They are not experimental
benchmarks and do not establish predictive accuracy.


### Example 1: Case 1 Baseline (With BOTH H2S & NO2 at -25 °C, 25 bar)


In [5]:
feed_case1 = {'H2S': 10.0, 'SO2': 10.0, 'NO2': 10.0, 'O2': 10.0, 'H2O': 10.0}
df1_t1, df1_t2 = generate_tables_for_case("Case 1: Baseline with Both H2S & NO2", feed_case1, T_K=248.15, P_bar=25.0)



CASE 1: BASELINE WITH BOTH H2S & NO2 (T=-25.0 °C, P=25.0 bar, material=carbon_steel)

TABLE 1: ILLUSTRATIVE 10-HOUR SPECIES CONCENTRATIONS

TABLE 2: ILLUSTRATIVE RATE-LAW SUMMARY


,Time (h),H2S (ppm),SO2 (ppm),NO2 (ppm),NO (ppm),O2 (ppm),H2O (ppm),H2SO4 (ppm),HNO3 (ppm),S8 (ppm),NH3 (ppm)
0,0.0,10.000000,10.000000,10.000000,0.000000,10.000000,10.000000,0.000000,0.0,0.000000,0.000000
1,1.0,0.166689,19.668831,0.007896,3.495342,8.287193,9.925286,0.162883,0.0,0.000200,6.496762
2,2.0,0.000000,19.757971,0.807377,2.648664,7.657675,9.943651,0.240410,0.0,0.000202,6.543959
3,3.0,0.000000,19.739247,1.417366,2.038674,7.343319,9.924927,0.259134,0.0,0.000202,6.543959
4,4.0,0.000000,19.720885,1.788694,1.667347,7.148474,9.906566,0.277495,0.0,0.000202,6.543959
5,5.0,0.000000,19.702780,2.040715,1.415325,7.013411,9.888461,0.295600,0.0,0.000202,6.543959
6,6.0,0.000000,19.684874,2.223893,1.232147,6.912869,9.870554,0.313507,0.0,0.000202,6.543959
7,7.0,0.000000,19.667130,2.363490,1.092550,6.834198,9.852810,0.331251,0.0,0.000202,6.543959
8,8.0,0.000000,19.649525,2.473644,0.982396,6.770319,9.835205,0.348856,0.0,0.000202,6.543959
9,9.0,0.000000,19.632042,2.562919,0.893122,6.716940,9.817723,0.366338,0.0,0.000202,6.543959


,ID,Net reaction,Ea (kJ/mol),Keq,k_forward,initial rate (kmol/m3/s),initial rate (ppm/h)
0,R1,SO2 + 0.5 O2 + H2O -> H2SO4,30.0,1.5284e+32,9.3484e-02,7.3639e-11,1.1030e-02
1,R2,H2S + 3 NO2 -> SO2 + H2O + 3 NO,30.0,5.6861e+83,4.8444e+03,2.5254e-04,3.7828e+04
2,R3A,SO2 + NO2 + H2O -> H2SO4 + NO,35.0,5.6738e+24,1.6569e-03,1.9722e-14,2.9541e-06
3,R3B,SO2 + 0.5 O2 + H2O -> H2SO4 (H2S/NO2 co-cataly...,18.0,irreversible,3.2522e+05,3.8709e-06,5.7982e+02
4,R4,2 NO + O2 -> 2 NO2,-4.4,7.2568e+14,4.2319e+03,0.0000e+00,0.0000e+00
5,R5,3 NO2 + H2O -> 2 HNO3 + NO,28.0,5.0811e-05,3.0650e+00,8.3296e-15,1.2477e-06
6,R6,H2S + 1.5 O2 -> SO2 + H2O,45.0,1.1116e+106,1.6859e-04,1.3280e-13,1.9892e-05
7,R7,5 H2S + 6 NO + 4 H2O -> 6 NH3 + 5 SO2,12.0,irreversible,5.9583e+03,0.0000e+00,0.0000e+00
8,R8,H2S + 0.5 O2 -> 1/8 S8 + H2O,42.0,irreversible,2.1648e-05,7.4687e-11,1.1187e-02


### Example 2: Case 2 (Without H2S at -25 °C, 25 bar)


In [6]:
feed_case2 = {'H2S': 0.0, 'SO2': 10.0, 'NO2': 10.0, 'O2': 10.0, 'H2O': 10.0}
df2_t1, df2_t2 = generate_tables_for_case("Case 2: Without H2S", feed_case2, T_K=248.15, P_bar=25.0)



CASE 2: WITHOUT H2S (T=-25.0 °C, P=25.0 bar, material=carbon_steel)

TABLE 1: ILLUSTRATIVE 10-HOUR SPECIES CONCENTRATIONS

TABLE 2: ILLUSTRATIVE RATE-LAW SUMMARY


,Time (h),H2S (ppm),SO2 (ppm),NO2 (ppm),NO (ppm),O2 (ppm),H2O (ppm),H2SO4 (ppm),HNO3 (ppm),S8 (ppm),NH3 (ppm)
0,0.0,0.0,10.000000,10.000000,0.000000,10.000000,10.000000,0.000000,0.000000,0.0,0.0
1,1.0,0.0,9.988980,9.999993,0.000004,9.994492,9.988979,0.011020,0.000002,0.0,0.0
2,2.0,0.0,9.977988,9.999987,0.000008,9.988997,9.977986,0.022012,0.000005,0.0,0.0
3,3.0,0.0,9.967023,9.999980,0.000013,9.983516,9.967019,0.032977,0.000007,0.0,0.0
4,4.0,0.0,9.956085,9.999973,0.000017,9.978048,9.956080,0.043915,0.000010,0.0,0.0
5,5.0,0.0,9.945174,9.999967,0.000021,9.972594,9.945168,0.054826,0.000012,0.0,0.0
6,6.0,0.0,9.934290,9.999960,0.000025,9.967154,9.934282,0.065710,0.000015,0.0,0.0
7,7.0,0.0,9.923432,9.999953,0.000029,9.961726,9.923423,0.076568,0.000017,0.0,0.0
8,8.0,0.0,9.912601,9.999947,0.000033,9.956312,9.912591,0.087399,0.000020,0.0,0.0
9,9.0,0.0,9.901797,9.999940,0.000037,9.950912,9.901786,0.098203,0.000022,0.0,0.0


,ID,Net reaction,Ea (kJ/mol),Keq,k_forward,initial rate (kmol/m3/s),initial rate (ppm/h)
0,R1,SO2 + 0.5 O2 + H2O -> H2SO4,30.0,1.5284e+32,9.3484e-02,7.3639e-11,1.1030e-02
1,R2,H2S + 3 NO2 -> SO2 + H2O + 3 NO,30.0,5.6861e+83,4.8444e+03,0.0000e+00,0.0000e+00
2,R3A,SO2 + NO2 + H2O -> H2SO4 + NO,35.0,5.6738e+24,1.6569e-03,1.9722e-14,2.9541e-06
3,R3B,SO2 + 0.5 O2 + H2O -> H2SO4 (H2S/NO2 co-cataly...,18.0,irreversible,3.2522e+05,0.0000e+00,0.0000e+00
4,R4,2 NO + O2 -> 2 NO2,-4.4,7.2568e+14,4.2319e+03,0.0000e+00,0.0000e+00
5,R5,3 NO2 + H2O -> 2 HNO3 + NO,28.0,5.0811e-05,3.0650e+00,8.3296e-15,1.2477e-06
6,R6,H2S + 1.5 O2 -> SO2 + H2O,45.0,1.1116e+106,1.6859e-04,0.0000e+00,0.0000e+00
7,R7,5 H2S + 6 NO + 4 H2O -> 6 NH3 + 5 SO2,12.0,irreversible,5.9583e+03,0.0000e+00,0.0000e+00
8,R8,H2S + 0.5 O2 -> 1/8 S8 + H2O,42.0,irreversible,2.1648e-05,0.0000e+00,0.0000e+00


### Example 3: Case 3 (Without NO2 at -25 °C, 25 bar)


In [7]:
feed_case3 = {'H2S': 10.0, 'SO2': 10.0, 'NO2': 0.0, 'O2': 10.0, 'H2O': 10.0}
df3_t1, df3_t2 = generate_tables_for_case("Case 3: Without NO2", feed_case3, T_K=248.15, P_bar=25.0)



CASE 3: WITHOUT NO2 (T=-25.0 °C, P=25.0 bar, material=carbon_steel)

TABLE 1: ILLUSTRATIVE 10-HOUR SPECIES CONCENTRATIONS

TABLE 2: ILLUSTRATIVE RATE-LAW SUMMARY


,Time (h),H2S (ppm),SO2 (ppm),NO2 (ppm),NO (ppm),O2 (ppm),H2O (ppm),H2SO4 (ppm),HNO3 (ppm),S8 (ppm),NH3 (ppm)
0,0.0,10.000000,10.000000,0.0,0.0,10.000000,10.000000,0.000000,0.0,0.000000,0.0
1,1.0,9.988802,9.988999,0.0,0.0,9.988871,10.000176,0.011021,0.0,0.001397,0.0
2,2.0,9.977623,9.978015,0.0,0.0,9.977760,10.000352,0.022024,0.0,0.002792,0.0
3,3.0,9.966463,9.967050,0.0,0.0,9.966667,10.000527,0.033010,0.0,0.004185,0.0
4,4.0,9.955322,9.956102,0.0,0.0,9.955593,10.000701,0.043977,0.0,0.005575,0.0
5,5.0,9.944199,9.945173,0.0,0.0,9.944538,10.000875,0.054926,0.0,0.006963,0.0
6,6.0,9.933095,9.934261,0.0,0.0,9.933500,10.001048,0.065858,0.0,0.008348,0.0
7,7.0,9.922009,9.923367,0.0,0.0,9.922481,10.001220,0.076771,0.0,0.009732,0.0
8,8.0,9.910942,9.912491,0.0,0.0,9.911480,10.001391,0.087667,0.0,0.011113,0.0
9,9.0,9.899893,9.901632,0.0,0.0,9.900498,10.001562,0.098545,0.0,0.012491,0.0


,ID,Net reaction,Ea (kJ/mol),Keq,k_forward,initial rate (kmol/m3/s),initial rate (ppm/h)
0,R1,SO2 + 0.5 O2 + H2O -> H2SO4,30.0,1.5284e+32,9.3484e-02,7.3639e-11,1.1030e-02
1,R2,H2S + 3 NO2 -> SO2 + H2O + 3 NO,30.0,5.6861e+83,4.8444e+03,0.0000e+00,0.0000e+00
2,R3A,SO2 + NO2 + H2O -> H2SO4 + NO,35.0,5.6738e+24,1.6569e-03,0.0000e+00,0.0000e+00
3,R3B,SO2 + 0.5 O2 + H2O -> H2SO4 (H2S/NO2 co-cataly...,18.0,irreversible,3.2522e+05,0.0000e+00,0.0000e+00
4,R4,2 NO + O2 -> 2 NO2,-4.4,7.2568e+14,4.2319e+03,0.0000e+00,0.0000e+00
5,R5,3 NO2 + H2O -> 2 HNO3 + NO,28.0,5.0811e-05,3.0650e+00,0.0000e+00,0.0000e+00
6,R6,H2S + 1.5 O2 -> SO2 + H2O,45.0,1.1116e+106,1.6859e-04,1.3280e-13,1.9892e-05
7,R7,5 H2S + 6 NO + 4 H2O -> 6 NH3 + 5 SO2,12.0,irreversible,5.9583e+03,0.0000e+00,0.0000e+00
8,R8,H2S + 0.5 O2 -> 1/8 S8 + H2O,42.0,irreversible,2.1648e-05,7.4687e-11,1.1187e-02


### Example 4: Case 4 (High Water 675 ppm & High NO2 72 ppm at 25 °C, 100 bar)


In [8]:
feed_case4 = {'H2O': 675.0, 'NO2': 72.0, 'SO2': 10.0, 'O2': 10.0, 'H2S': 0.0}
df4_t1, df4_t2 = generate_tables_for_case("Case 4: High Water & NO2 Screening Case", feed_case4, T_K=298.15, P_bar=100.0)



CASE 4: HIGH WATER & NO2 SCREENING CASE (T=25.0 °C, P=100.0 bar, material=carbon_steel)

TABLE 1: ILLUSTRATIVE 10-HOUR SPECIES CONCENTRATIONS

TABLE 2: ILLUSTRATIVE RATE-LAW SUMMARY


,Time (h),H2S (ppm),SO2 (ppm),NO2 (ppm),NO (ppm),O2 (ppm),H2O (ppm),H2SO4 (ppm),HNO3 (ppm),S8 (ppm),NH3 (ppm)
0,0.0,0.0,10.000000,72.000000,0.000000,10.000000,675.000000,0.000000,0.000000,0.0,0.0
1,1.0,0.0,2.197043,71.408993,0.212035,6.109796,667.007557,7.802957,0.378973,0.0,0.0
2,2.0,0.0,0.594356,70.896559,0.383899,5.309242,665.234585,9.405644,0.719543,0.0,0.0
3,3.0,0.0,0.169689,70.509343,0.508511,5.093564,664.678616,9.830311,0.982145,0.0,0.0
4,4.0,0.0,0.049218,70.266171,0.581932,5.027601,664.473270,9.950782,1.151897,0.0,0.0
5,5.0,0.0,0.014351,70.134967,0.616316,5.003155,664.389993,9.985649,1.248717,0.0,0.0
6,6.0,0.0,0.004194,70.070417,0.627766,4.990525,664.353285,9.995806,1.301817,0.0,0.0
7,7.0,0.0,0.001227,70.039895,0.627715,4.981373,664.335032,9.998773,1.332391,0.0,0.0
8,8.0,0.0,0.000360,70.025395,0.622433,4.973353,664.324274,9.999640,1.352172,0.0,0.0
9,9.0,0.0,0.000105,70.018116,0.614969,4.965808,664.316648,9.999895,1.366915,0.0,0.0


,ID,Net reaction,Ea (kJ/mol),Keq,k_forward,initial rate (kmol/m3/s),initial rate (ppm/h)
0,R1,SO2 + 0.5 O2 + H2O -> H2SO4,30.0,6.1224e+26,2.7746e+00,1.0112e-07,1.7617e+01
1,R2,H2S + 3 NO2 -> SO2 + H2O + 3 NO,30.0,5.1176e+69,5.5492e+04,0.0000e+00,0.0000e+00
2,R3A,SO2 + NO2 + H2O -> H2SO4 + NO,35.0,4.0052e+20,7.3835e-02,2.7147e-10,4.7294e-02
3,R3B,SO2 + 0.5 O2 + H2O -> H2SO4 (H2S/NO2 co-cataly...,18.0,irreversible,1.4046e+06,0.0000e+00,0.0000e+00
4,R4,2 NO + O2 -> 2 NO2,-4.4,2.3366e+12,2.9579e+03,0.0000e+00,0.0000e+00
5,R5,3 NO2 + H2O -> 2 HNO3 + NO,28.0,2.6673e-04,2.9842e+01,1.1166e-09,1.9453e-01
6,R6,H2S + 1.5 O2 -> SO2 + H2O,45.0,1.8279e+88,6.5360e-03,0.0000e+00,0.0000e+00
7,R7,5 H2S + 6 NO + 4 H2O -> 6 NH3 + 5 SO2,12.0,irreversible,1.5802e+04,0.0000e+00,0.0000e+00
8,R8,H2S + 0.5 O2 -> 1/8 S8 + H2O,42.0,irreversible,6.5767e-04,0.0000e+00,0.0000e+00


### Example 5: Oxidant-Free Stream (100 ppm H2O & 100 ppm H2S at 25 °C, 100 bar)


In [9]:
feed_oxidant_free = {'H2O': 100.0, 'H2S': 100.0, 'SO2': 0.0, 'NO2': 0.0, 'O2': 0.0}
df_off_t1, df_off_t2 = generate_tables_for_case("Oxidant-Free Stream", feed_oxidant_free, T_K=298.15, P_bar=100.0)



OXIDANT-FREE STREAM (T=25.0 °C, P=100.0 bar, material=carbon_steel)

TABLE 1: ILLUSTRATIVE 10-HOUR SPECIES CONCENTRATIONS

TABLE 2: ILLUSTRATIVE RATE-LAW SUMMARY


,Time (h),H2S (ppm),SO2 (ppm),NO2 (ppm),NO (ppm),O2 (ppm),H2O (ppm),H2SO4 (ppm),HNO3 (ppm),S8 (ppm),NH3 (ppm)
0,0.0,100.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0
1,1.0,100.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0
2,2.0,100.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0
3,3.0,100.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0
4,4.0,100.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0
5,5.0,100.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0
6,6.0,100.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0
7,7.0,100.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0
8,8.0,100.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0
9,9.0,100.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0


,ID,Net reaction,Ea (kJ/mol),Keq,k_forward,initial rate (kmol/m3/s),initial rate (ppm/h)
0,R1,SO2 + 0.5 O2 + H2O -> H2SO4,30.0,6.1224e+26,2.4930e+00,0.0000e+00,0.0000e+00
1,R2,H2S + 3 NO2 -> SO2 + H2O + 3 NO,30.0,5.1176e+69,5.5492e+04,0.0000e+00,0.0000e+00
2,R3A,SO2 + NO2 + H2O -> H2SO4 + NO,35.0,4.0052e+20,6.6341e-02,0.0000e+00,0.0000e+00
3,R3B,SO2 + 0.5 O2 + H2O -> H2SO4 (H2S/NO2 co-cataly...,18.0,irreversible,1.4046e+06,0.0000e+00,0.0000e+00
4,R4,2 NO + O2 -> 2 NO2,-4.4,2.3366e+12,2.9579e+03,0.0000e+00,0.0000e+00
5,R5,3 NO2 + H2O -> 2 HNO3 + NO,28.0,2.6673e-04,2.9842e+01,0.0000e+00,0.0000e+00
6,R6,H2S + 1.5 O2 -> SO2 + H2O,45.0,1.8279e+88,6.5360e-03,0.0000e+00,0.0000e+00
7,R7,5 H2S + 6 NO + 4 H2O -> 6 NH3 + 5 SO2,12.0,irreversible,1.5802e+04,0.0000e+00,0.0000e+00
8,R8,H2S + 0.5 O2 -> 1/8 S8 + H2O,42.0,irreversible,6.5767e-04,0.0000e+00,0.0000e+00


### Example 6: Pipeline Transport Stream (11 ppm H2O, 69 ppm SO2, 33 ppm NO2, 180 ppm O2 at 25 °C, 70 bar)


In [10]:
feed_pipe = {'H2O': 11.0, 'SO2': 69.0, 'NO2': 33.0, 'O2': 180.0, 'H2S': 0.0}
df_pipe_t1, df_pipe_t2 = generate_tables_for_case("High-O2 Pipeline Transport Stream", feed_pipe, T_K=298.15, P_bar=70.0)



HIGH-O2 PIPELINE TRANSPORT STREAM (T=25.0 °C, P=70.0 bar, material=carbon_steel)

TABLE 1: ILLUSTRATIVE 10-HOUR SPECIES CONCENTRATIONS

TABLE 2: ILLUSTRATIVE RATE-LAW SUMMARY


,Time (h),H2S (ppm),SO2 (ppm),NO2 (ppm),NO (ppm),O2 (ppm),H2O (ppm),H2SO4 (ppm),HNO3 (ppm),S8 (ppm),NH3 (ppm)
0,0.0,0.0,69.000000,33.000000,0.000000,180.000000,11.000000,0.000000,0.000000,0.0,0.0
1,1.0,0.0,66.466586,32.998674,0.000912,178.733646,8.466379,2.533414,0.000414,0.0,0.0
2,2.0,0.0,64.577287,32.997668,0.001598,177.789259,6.576920,4.422713,0.000735,0.0,0.0
3,3.0,0.0,63.145139,32.996896,0.002119,177.073383,5.144647,5.854861,0.000985,0.0,0.0
4,4.0,0.0,62.046111,32.996301,0.002519,176.524020,4.045521,6.953889,0.001181,0.0,0.0
5,5.0,0.0,61.194786,32.995838,0.002827,176.098473,3.194119,7.805214,0.001335,0.0,0.0
6,6.0,0.0,60.530557,32.995479,0.003064,175.766446,2.529829,8.469443,0.001457,0.0,0.0
7,7.0,0.0,60.009389,32.995198,0.003248,175.505930,2.008612,8.990611,0.001554,0.0,0.0
8,8.0,0.0,59.598670,32.994980,0.003389,175.300622,1.597855,9.401330,0.001631,0.0,0.0
9,9.0,0.0,59.273874,32.994810,0.003498,175.138263,1.273028,9.726126,0.001692,0.0,0.0


,ID,Net reaction,Ea (kJ/mol),Keq,k_forward,initial rate (kmol/m3/s),initial rate (ppm/h)
0,R1,SO2 + 0.5 O2 + H2O -> H2SO4,30.0,6.1224e+26,1.1046e+00,1.5488e-08,2.9407e+00
1,R2,H2S + 3 NO2 -> SO2 + H2O + 3 NO,30.0,5.1176e+69,5.5492e+04,0.0000e+00,0.0000e+00
2,R3A,SO2 + NO2 + H2O -> H2SO4 + NO,35.0,4.0052e+20,2.9395e-02,4.3024e-12,8.1691e-04
3,R3B,SO2 + 0.5 O2 + H2O -> H2SO4 (H2S/NO2 co-cataly...,18.0,irreversible,1.4046e+06,0.0000e+00,0.0000e+00
4,R4,2 NO + O2 -> 2 NO2,-4.4,2.3366e+12,2.9579e+03,0.0000e+00,0.0000e+00
5,R5,3 NO2 + H2O -> 2 HNO3 + NO,28.0,2.6673e-04,2.9842e+01,1.2417e-12,2.3576e-04
6,R6,H2S + 1.5 O2 -> SO2 + H2O,45.0,1.8279e+88,6.5360e-03,0.0000e+00,0.0000e+00
7,R7,5 H2S + 6 NO + 4 H2O -> 6 NH3 + 5 SO2,12.0,irreversible,1.5802e+04,0.0000e+00,0.0000e+00
8,R8,H2S + 0.5 O2 -> 1/8 S8 + H2O,42.0,irreversible,6.5767e-04,0.0000e+00,0.0000e+00


### Example 7: High H2S Pipeline Stream (60 ppm H2S, 10 ppm others at 25 °C, 100 bar)


In [11]:
feed_high_h2s = {'H2S': 60.0, 'SO2': 10.0, 'NO2': 10.0, 'O2': 10.0, 'H2O': 10.0}
df_h2s_t1, df_h2s_t2 = generate_tables_for_case("High-H2S Pipeline Transport Stream", feed_high_h2s, T_K=298.15, P_bar=100.0)



HIGH-H2S PIPELINE TRANSPORT STREAM (T=25.0 °C, P=100.0 bar, material=carbon_steel)

TABLE 1: ILLUSTRATIVE 10-HOUR SPECIES CONCENTRATIONS

TABLE 2: ILLUSTRATIVE RATE-LAW SUMMARY


,Time (h),H2S (ppm),SO2 (ppm),NO2 (ppm),NO (ppm),O2 (ppm),H2O (ppm),H2SO4 (ppm),HNO3 (ppm),S8 (ppm),NH3 (ppm)
0,0.0,60.000000,10.000000,10.0,0.0,10.000000,10.000000,0.000000,0.0,0.000000,0.0
1,1.0,46.843125,21.522003,0.0,0.0,9.151712,7.991643,0.165231,0.0,0.183705,10.0
2,2.0,45.479250,21.350674,0.0,0.0,8.380617,9.181861,0.338889,0.0,0.353898,10.0
3,3.0,44.212376,21.166222,0.0,0.0,7.651986,10.262303,0.525320,0.0,0.512010,10.0
4,4.0,43.035876,20.972788,0.0,0.0,6.964506,11.243694,0.720430,0.0,0.658863,10.0
5,5.0,41.943717,20.773918,0.0,0.0,6.316871,12.135570,0.920713,0.0,0.795207,10.0
6,6.0,40.930404,20.572629,0.0,0.0,5.707790,12.946408,1.123189,0.0,0.921722,10.0
7,7.0,39.990936,20.371480,0.0,0.0,5.135992,13.683733,1.325331,0.0,1.039032,10.0
8,8.0,39.120763,20.172621,0.0,0.0,4.600237,14.354222,1.525015,0.0,1.147700,10.0
9,9.0,38.315744,19.977861,0.0,0.0,4.099324,14.963798,1.720458,0.0,1.248242,10.0


,ID,Net reaction,Ea (kJ/mol),Keq,k_forward,initial rate (kmol/m3/s),initial rate (ppm/h)
0,R1,SO2 + 0.5 O2 + H2O -> H2SO4,30.0,6.1224e+26,1.0709e+00,5.7821e-10,1.0073e-01
1,R2,H2S + 3 NO2 -> SO2 + H2O + 3 NO,30.0,5.1176e+69,5.5492e+04,1.2831e-02,2.2354e+06
2,R3A,SO2 + NO2 + H2O -> H2SO4 + NO,35.0,4.0052e+20,2.8497e-02,2.1559e-13,3.7558e-05
3,R3B,SO2 + 0.5 O2 + H2O -> H2SO4 (H2S/NO2 co-cataly...,18.0,irreversible,1.4046e+06,2.6030e-05,4.5347e+03
4,R4,2 NO + O2 -> 2 NO2,-4.4,2.3366e+12,2.9579e+03,0.0000e+00,0.0000e+00
5,R5,3 NO2 + H2O -> 2 HNO3 + NO,28.0,2.6673e-04,2.9842e+01,4.4319e-14,7.7210e-06
6,R6,H2S + 1.5 O2 -> SO2 + H2O,45.0,1.8279e+88,6.5360e-03,2.1175e-11,3.6889e-03
7,R7,5 H2S + 6 NO + 4 H2O -> 6 NH3 + 5 SO2,12.0,irreversible,1.5802e+04,0.0000e+00,0.0000e+00
8,R8,H2S + 0.5 O2 -> 1/8 S8 + H2O,42.0,irreversible,6.5767e-04,1.0854e-08,1.8909e+00
